# Dedekind Python DSL Sketch: Executable Demo
This notebook is a first executable design sketch aligned with issue #241. It demonstrates a Pythonic layer and a symbolic-flavored layer while keeping runtime behavior deterministic for CI.

## Important scope note
This is a **prototype notebook shim**, not the final `dedekind` API surface. It imports `dedekind` and uses today's MVP facade where possible, while sketching the proposed DSL ergonomics in executable Python.

In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing pyproject.toml")

repo_root = find_repo_root(Path.cwd())

try:
    import dedekind
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "dedekind is not installed in this notebook kernel. "
        f"From {repo_root}, run `python -m pip install -e .` "
        "or use `make integration-test` for the CI-style execution path."
    ) from err

print("dsl-demo: dedekind import ok")

In [ ]:
class DemoSet:
    """Prototype-only helper for DSL ergonomics demo."""

    def __init__(self, values):
        self._values = list(values)

    @staticmethod
    def from_iterable(values):
        return DemoSet(values)

    def where(self, pred):
        return DemoSet(v for v in self._values if pred(v))

    def image(self, fn):
        return DemoSet(fn(v) for v in self._values)

    def realize(self, *, ordered=True):
        if ordered:
            return dedekind.ordered_set_roundtrip(self._values)
        return dedekind.unordered_set_roundtrip(self._values)

    def union(self, other):
        return DemoSet(set(self._values) | set(other._values))

    def intersection(self, other):
        return DemoSet(set(self._values) & set(other._values))

    def difference(self, other):
        return DemoSet(set(self._values) - set(other._values))

    def symdiff(self, other):
        return DemoSet(set(self._values) ^ set(other._values))

    def product(self, other):
        return DemoSet((a, b) for a in self._values for b in other._values)

    def __or__(self, other):
        return self.union(other)

    def __and__(self, other):
        return self.intersection(other)

    def __sub__(self, other):
        return self.difference(other)

    def __xor__(self, other):
        return self.symdiff(other)

    def __mul__(self, other):
        return self.product(other)

    def __repr__(self):
        return f"DemoSet({self._values})"


class Sym:
    @staticmethod
    def N(limit=32):
        return DemoSet(range(limit))


sym = Sym()
print("dsl-demo: shim types ready")

In [ ]:
# Easy tier sketch
A = DemoSet.from_iterable([1, 2, 3])
B = DemoSet.from_iterable([2, 3, 4])

U = A | B
I = A & B
D = A - B
S = DemoSet.from_iterable(range(20)).where(lambda x: x % 2 == 0).image(lambda x: x * x)

easy_union = U.realize()
easy_intersection = I.realize()
easy_difference = D.realize()
easy_squares = S.realize()

print("easy.union      ->", easy_union)
print("easy.intersection->", easy_intersection)
print("easy.difference  ->", easy_difference)
print("easy.squares     ->", easy_squares)

assert easy_union == [1, 2, 3, 4]
assert easy_intersection == [2, 3]
assert easy_difference == [1]
assert easy_squares[:6] == [0, 4, 16, 36, 64, 100]

In [ ]:
# Expert tier sketch (symbolic-flavored, prototype)
N = sym.N(limit=24)
E = N.where(lambda alpha: alpha % 2 == 0)
O = N.where(lambda alpha: alpha % 2 == 1)

expert_union = (E | O).realize()
expert_even = E.realize()
expert_odd = O.realize()

print("expert.N[:12]   ->", expert_union[:12])
print("expert.evens[:8] ->", expert_even[:8])
print("expert.odds[:8]  ->", expert_odd[:8])

assert expert_union[:10] == list(range(10))
assert expert_even[:6] == [0, 2, 4, 6, 8, 10]
assert expert_odd[:6] == [1, 3, 5, 7, 9, 11]
print("notebook-03-ok")